In [ ]:
import pandas as pd
import numpy as np

# Function 1: Classify Lane Changes
def classify_lane_changes(tracks_df):
    tracks_df['laneChangeType'] = 0
    def lane_change_logic(df):
        lane_diff = df['laneId'].diff()
        direction = df['drivingDirection'].iloc[0]
        lane_change = np.where(
            (lane_diff > 0) & (direction == 1), 1,
            np.where((lane_diff < 0) & (direction == 1), -1,
                     np.where((lane_diff > 0) & (direction == 2), -1,
                              np.where((lane_diff < 0) & (direction == 2), 1, 0)))
        )
        return lane_change
    tracks_df['laneChangeType'] = tracks_df.groupby('id', group_keys=False).apply(lane_change_logic)
    return tracks_df

# Function 2: Calculate Acceleration
def calculate_acceleration(tracks_df):
    tracks_df['xAcceleration'] = tracks_df.groupby('id')['xVelocity'].diff().fillna(0)
    tracks_df['yAcceleration'] = tracks_df.groupby('id')['yVelocity'].diff().fillna(0)
    return tracks_df

# Function 3: Calculate Relative Positions and Velocities (with ego vehicle reference)
def calculate_relative_positions_and_velocities(tracks_df, ego_id=5):
    def calculate_relative_data(row):
        # Assign relative positions in the desired order:
        surrounding_ids = [
            row['leftPrecedingId'], row['precedingId'], row['rightPrecedingId'],
            row['leftAlongsideId'], row['id'], row['rightAlongsideId'],
            row['leftFollowingId'], row['followingId'], row['rightFollowingId']
        ]
        surrounding_data = {}
        for i, s_id in enumerate(surrounding_ids):
            if s_id > 0:
                s_vehicle = tracks_df[tracks_df['id'] == s_id]
                if not s_vehicle.empty:
                    dx = s_vehicle['x'].values[0] - row['x']
                    dy = s_vehicle['y'].values[0] - row['y']
                    dvx = s_vehicle['xVelocity'].values[0] - row['xVelocity']
                    dvy = s_vehicle['yVelocity'].values[0] - row['yVelocity']
                    surrounding_data[f'vehicle_{i+1}_relative_x'] = dx
                    surrounding_data[f'vehicle_{i+1}_relative_y'] = dy
                    surrounding_data[f'vehicle_{i+1}_relative_vx'] = dvx
                    surrounding_data[f'vehicle_{i+1}_relative_vy'] = dvy
                else:
                    surrounding_data[f'vehicle_{i+1}_relative_x'] = -1
                    surrounding_data[f'vehicle_{i+1}_relative_y'] = -1
                    surrounding_data[f'vehicle_{i+1}_relative_vx'] = 0
                    surrounding_data[f'vehicle_{i+1}_relative_vy'] = 0
            else:
                surrounding_data[f'vehicle_{i+1}_relative_x'] = -1
                surrounding_data[f'vehicle_{i+1}_relative_y'] = -1
                surrounding_data[f'vehicle_{i+1}_relative_vx'] = 0
                surrounding_data[f'vehicle_{i+1}_relative_vy'] = 0

        # Add ego vehicle's absolute values but exclude acceleration
        surrounding_data['vehicle_5_relative_x'] = row['x']
        surrounding_data['vehicle_5_relative_y'] = row['y']
        surrounding_data['vehicle_5_relative_vx'] = row['xVelocity']
        surrounding_data['vehicle_5_relative_vy'] = row['yVelocity']

        return surrounding_data

    relative_data = tracks_df.apply(calculate_relative_data, axis=1)
    relative_data_df = pd.DataFrame(relative_data.tolist())
    tracks_df = pd.concat([tracks_df, relative_data_df], axis=1)
    return tracks_df

# Function 4: Reshape into a single column matrix
def reshape_to_vehicle_matrix(tracks_df):
    matrix_data = []
    for _, row in tracks_df.iterrows():
        vehicle_data = []
        for vehicle_num in range(1, 10):
            vehicle_features = [
                row.get(f"vehicle_{vehicle_num}_relative_x", -1),
                row.get(f"vehicle_{vehicle_num}_relative_y", -1),
                row.get(f"vehicle_{vehicle_num}_relative_vx", 0),
                row.get(f"vehicle_{vehicle_num}_relative_vy", 0),
            ]
            vehicle_data.append(vehicle_features)

        # Flatten into a single column
        matrix_data.append(np.array(vehicle_data).flatten())

    matrix_columns = [f"feature_{i+1}" for i in range(len(matrix_data[0]))]
    vehicle_matrix = pd.DataFrame(matrix_data, columns=matrix_columns)
    return vehicle_matrix

# Function 5: Fill Missing Values
def fill_missing_values(df):
    relative_position_columns = [col for col in df.columns if 'relative_x' in col or 'relative_y' in col]
    velocity_columns = [col for col in df.columns if 'relative_vx' in col or 'relative_vy' in col]

    ego_vx = df['xVelocity']
    ego_vy = df['yVelocity']

    for col in relative_position_columns:
        df[col] = df[col].fillna(-1)

    for col in velocity_columns:
        if 'relative_vx' in col:
            df[col] = df[col].fillna(ego_vx)
        elif 'relative_vy' in col:
            df[col] = df[col].fillna(ego_vy)

    return df

# Function 6: Assign Time Intervals
def assign_time_intervals(df, interval=75):
    df['time'] = 'T1'
    max_frame = df['frame'].max()
    time_labels = [f'T{i+1}' for i in range((max_frame // interval) + 1)]
    for i, label in enumerate(time_labels):
        start_frame = i * interval
        end_frame = start_frame + interval - 1
        df.loc[(df['frame'] >= start_frame) & (df['frame'] <= end_frame), 'time'] = label
    return df

# Main Execution
if __name__ == "__main__":
    # Load the meta and track data
    tracks_df = pd.read_csv('/content/dynamic_y_adjusted_tracks_new.csv')
    meta_df = pd.read_csv('01_tracksMeta.csv')

    # Merge meta features with tracks
    combined_df = tracks_df.merge(meta_df, on='id', how='left')

    # Classify lane changes
    combined_df = classify_lane_changes(combined_df)

    # Calculate acceleration
    combined_df = calculate_acceleration(combined_df)

    # Calculate relative positions and velocities
    combined_df = calculate_relative_positions_and_velocities(combined_df)

    # Fill missing values
    combined_df = fill_missing_values(combined_df)

    # Assign time intervals
    combined_df = assign_time_intervals(combined_df)

    # Reshape to vehicle matrix
    vehicle_matrix = reshape_to_vehicle_matrix(combined_df)

    # Add the matrix as a single column in the original DataFrame
    combined_df['vehicle_matrix'] = vehicle_matrix.apply(lambda row: row.tolist(), axis=1)

    # Save the final processed data
    combined_df.to_csv('processed_traffic_data_with_matrix_and_time.csv', index=False)

    print("Data preprocessing complete. File saved as 'processed_traffic_data_with_matrix_and_time.csv'.")

